In [2]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import lightgbm
from xgboost import XGBRegressor
import pandas as pd
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import *
from src.data_split import *
from src.metrics import *
from src.modeling import *


In [3]:
train_df = pd.read_csv('../data/train_test/holdout/train_full.csv', index_col='id', parse_dates=['timestamp'])
test_df = pd.read_csv('../data/train_test/holdout/holdout.csv', index_col='id', parse_dates=['timestamp'])

C:\Users\sokol\AppData\Local\Temp\ipykernel_24588\3531396417.py:1: DtypeWarning: Columns (0: old_education_build_share, 1: provision_doctors) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv('../data/train_test/holdout/train_full.csv', index_col='id', parse_dates=['timestamp'])
C:\Users\sokol\AppData\Local\Temp\ipykernel_24588\3531396417.py:2: DtypeWarning: Columns (0: load_of_teachers_school_per_teacher) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv('../data/train_test/holdout/holdout.csv', index_col='id', parse_dates=['timestamp'])


In [4]:
time_folds = get_time_folds()
models = []

In [5]:
model = CatBoostRegressor(
iterations=3000,
learning_rate=0.05,
depth=6,
loss_function="RMSE",
eval_metric="RMSE",
random_seed=42,
early_stopping_rounds=200,
verbose=200,
)


catboost_quality, oof_pred_catboost, catboost_models, features_import_cat, catb_features = run_boosting_cv(
    train_df=train_df,
    time_folds=time_folds,
    split_fold=split_fold,
    model_factory=model,
    model_name="catboost_baseline",
    booster="catboost",
    preprocessing_func=base_preprocessing,
)

catboost_quality

0:	learn: 0.6049682	test: 0.5822108	best: 0.5822108 (0)	total: 236ms	remaining: 11m 48s
200:	learn: 0.4626079	test: 0.4712833	best: 0.4712833 (200)	total: 16s	remaining: 3m 43s
400:	learn: 0.4253471	test: 0.4696774	best: 0.4696497 (396)	total: 31.1s	remaining: 3m 21s
600:	learn: 0.3912850	test: 0.4710062	best: 0.4692238 (429)	total: 46s	remaining: 3m 3s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.4692238153
bestIteration = 429

Shrink model to first 430 iterations.
0:	learn: 0.5965759	test: 0.6044838	best: 0.6044838 (0)	total: 112ms	remaining: 5m 35s
200:	learn: 0.4616152	test: 0.4789304	best: 0.4789304 (200)	total: 16.4s	remaining: 3m 48s
400:	learn: 0.4338423	test: 0.4772623	best: 0.4772623 (400)	total: 32.3s	remaining: 3m 29s
600:	learn: 0.4089249	test: 0.4802913	best: 0.4772623 (400)	total: 47.9s	remaining: 3m 11s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.4772622767
bestIteration = 400

Shrink model to first 401 iterations.
0:	lea

,model_name,fold_1,fold_2,fold_3,mean_rmsle,rmsle_std
0,catboost_baseline,0.469224,0.477262,0.439019,0.461835,0.020164


In [6]:
model = LGBMRegressor(
n_estimators=3000,
learning_rate=0.05,
num_leaves=31,
max_depth=-1,
min_child_samples=20,
subsample=0.8,
colsample_bytree=0.8,
objective="regression",
random_state=42,
n_jobs=-1
)

lightgbm_quality, oof_pred_lightgbm, lightgbm_models, features_import_light, lgb_features = run_boosting_cv(
    train_df=train_df,
    time_folds=time_folds,
    split_fold=split_fold,
    model_factory=model,
    model_name="lightgbm_baseline",
    booster="lightgbm",
    preprocessing_func=base_preprocessing,
)
lightgbm_quality

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.035476 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 42315
[LightGBM] [Info] Number of data points in the train set: 10160, number of used features: 391
[LightGBM] [Info] Start training from score 15.501434
Training until validation scores don't improve for 200 rounds
[200]	valid_0's rmse: 0.47746	valid_0's l2: 0.227968
Early stopping, best iteration is:
[115]	valid_0's rmse: 0.474571	valid_0's l2: 0.225217
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.046868 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 43152
[LightGBM] [Info] Number of data points in the train set: 15231, number of used features: 391
[LightGBM] [Info] Start training from score 15.535907
Training until validation scores don't improve for 200 rounds
[200]	valid_0's rmse: 0.485414	valid_0's l2:

,model_name,fold_1,fold_2,fold_3,mean_rmsle,rmsle_std
0,lightgbm_baseline,0.474571,0.484132,0.446759,0.468487,0.019415


In [7]:
model = XGBRegressor(
n_estimators=3000,
learning_rate=0.05,
max_depth=6,
min_child_weight=1,
subsample=0.8,
colsample_bytree=0.8,
objective="reg:squarederror",
eval_metric="rmse",
tree_method="hist",
enable_categorical=True,
early_stopping_rounds=200,
random_state=42,
n_jobs=-1,
)


xgboost_quality, oof_pred_xgboost, xgboost_models, features_import_xgb, xgb_features = run_boosting_cv(
    train_df=train_df,
    time_folds=time_folds,
    split_fold=split_fold,
    model_factory=model,
    model_name="xgboost_baseline",
    booster="xgboost",
    preprocessing_func=base_preprocessing,
)

xgboost_quality

[0]	validation_0-rmse:0.58066
[200]	validation_0-rmse:0.49040
[261]	validation_0-rmse:0.49542
[0]	validation_0-rmse:0.60063
[200]	validation_0-rmse:0.49544
[300]	validation_0-rmse:0.50220
[0]	validation_0-rmse:0.59722
[200]	validation_0-rmse:0.46107
[304]	validation_0-rmse:0.47379


,model_name,fold_1,fold_2,fold_3,mean_rmsle,rmsle_std
0,xgboost_baseline,0.482901,0.491692,0.455477,0.47669,0.018889


In [8]:
model = CatBoostRegressor(
iterations=3000,
learning_rate=0.05,
depth=6,
loss_function="RMSE",
eval_metric="RMSE",
random_seed=42,
early_stopping_rounds=200,
verbose=200,
)


catboost_eng_quality, oof_pred_catboost_eng, catboost_eng_models, features_import_cat_eng, catb_eng_features = run_boosting_cv(
    train_df=train_df,
    time_folds=time_folds,
    split_fold=split_fold,
    model_factory=model,
    model_name="catboost_eng",
    booster="catboost",
    preprocessing_func=prep_with_features_eng,
)

catboost_eng_quality

0:	learn: 0.6050475	test: 0.5823720	best: 0.5823720 (0)	total: 97.9ms	remaining: 4m 53s
200:	learn: 0.4613451	test: 0.4674028	best: 0.4673556 (198)	total: 15.2s	remaining: 3m 32s
400:	learn: 0.4252100	test: 0.4678061	best: 0.4666759 (236)	total: 30.2s	remaining: 3m 15s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.4666759206
bestIteration = 236

Shrink model to first 237 iterations.
0:	learn: 0.5967628	test: 0.6021311	best: 0.6021311 (0)	total: 123ms	remaining: 6m 8s
200:	learn: 0.4623757	test: 0.4774855	best: 0.4774830 (199)	total: 16.3s	remaining: 3m 47s
400:	learn: 0.4331304	test: 0.4755781	best: 0.4755533 (397)	total: 32s	remaining: 3m 27s
600:	learn: 0.4082472	test: 0.4774237	best: 0.4751948 (411)	total: 47.9s	remaining: 3m 11s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.4751947919
bestIteration = 411

Shrink model to first 412 iterations.
0:	learn: 0.5972996	test: 0.5992900	best: 0.5992900 (0)	total: 109ms	remaining: 5m 27s
200:	lea

,model_name,fold_1,fold_2,fold_3,mean_rmsle,rmsle_std
0,catboost_eng,0.466676,0.475195,0.436673,0.459514,0.020235


In [10]:
cat_boost_eng_features_importance_df = pd.DataFrame({
    'features': catb_eng_features,
    'importance_val1': features_import_cat_eng[0],
    'importance_val2': features_import_cat_eng[1],
    'importance_val3': features_import_cat_eng[2]
})
cat_boost_eng_features_importance_df['mean_importance_catboost'] = (cat_boost_eng_features_importance_df['importance_val1'] + cat_boost_eng_features_importance_df['importance_val2'] + cat_boost_eng_features_importance_df['importance_val3'])/3
cat_boost_eng_features_importance_df = cat_boost_eng_features_importance_df.sort_values('mean_importance_catboost', ascending=False)
cat_boost_eng_features_importance_df['cat_imp_ranks'] = cat_boost_eng_features_importance_df['mean_importance_catboost'].rank(ascending=False)

ValueError: All arrays must be of the same length

In [ ]:
model = LGBMRegressor(
n_estimators=3000,
learning_rate=0.05,
num_leaves=31,
max_depth=-1,
min_child_samples=20,
subsample=0.8,
colsample_bytree=0.8,
objective="regression",
random_state=42,
n_jobs=-1
)

lightgbm_eng_quality, oof_pred_lightgbm_eng, lightgbm_eng_models, features_import_light_eng, lgb_eng_features = run_boosting_cv(
    train_df=train_df,
    time_folds=time_folds,
    split_fold=split_fold,
    model_factory=model,
    model_name="lightgbm_eng",
    booster="lightgbm",
    preprocessing_func=prep_with_features_eng,
)
lightgbm_eng_quality

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013886 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 43281
[LightGBM] [Info] Number of data points in the train set: 10160, number of used features: 402
[LightGBM] [Info] Start training from score 15.501434
Training until validation scores don't improve for 200 rounds
[200]	valid_0's rmse: 0.477838	valid_0's l2: 0.228329
Early stopping, best iteration is:
[129]	valid_0's rmse: 0.475595	valid_0's l2: 0.22619
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.026034 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 44497
[LightGBM] [Info] Number of data points in the train set: 15231, number of used features: 402
[LightGBM] [Info] Start training from score 15.535907
Training until validation scores don't improve for 200 rounds
[200]	valid_0's rmse: 0.483636	valid_0's l2:

,model_name,fold_1,fold_2,fold_3,mean_rmsle,rmsle_std
0,lightgbm_eng,0.475595,0.482197,0.444557,0.46745,0.020099


In [ ]:
lgb_boost_eng_features_importance_df = pd.DataFrame({
    'features': lgb_eng_features,
    'importance_val1': features_import_light_eng[0],
    'importance_val2': features_import_light_eng[1],
    'importance_val3': features_import_light_eng[2]
})
lgb_boost_eng_features_importance_df['mean_importance_lgb'] = (lgb_boost_eng_features_importance_df['importance_val1'] + lgb_boost_eng_features_importance_df['importance_val2'] + lgb_boost_eng_features_importance_df['importance_val3'])/3
lgb_boost_eng_features_importance_df = lgb_boost_eng_features_importance_df.sort_values('mean_importance_lgb', ascending=False)
lgb_boost_eng_features_importance_df['lgb_imp_ranks'] = lgb_boost_eng_features_importance_df['mean_importance_lgb'].rank(ascending=False)

In [ ]:
model = XGBRegressor(
n_estimators=3000,
learning_rate=0.05,
max_depth=6,
min_child_weight=1,
subsample=0.8,
colsample_bytree=0.8,
objective="reg:squarederror",
eval_metric="rmse",
tree_method="hist",
enable_categorical=True,
early_stopping_rounds=200,
random_state=42,
n_jobs=-1,
)


xgboost_eng_quality, oof_pred_xgboost_eng, xgboost_eng_models, features_import_xgb_eng, xgb_eng_features = run_boosting_cv(
    train_df=train_df,
    time_folds=time_folds,
    split_fold=split_fold,
    model_factory=model,
    model_name="xgboost_eng",
    booster="xgboost",
    preprocessing_func=prep_with_features_eng,
)

xgboost_quality

[0]	validation_0-rmse:0.58347
[200]	validation_0-rmse:0.49157
[261]	validation_0-rmse:0.49742
[0]	validation_0-rmse:0.60196
[200]	validation_0-rmse:0.48927
[309]	validation_0-rmse:0.49540
[0]	validation_0-rmse:0.59815
[200]	validation_0-rmse:0.45789
[312]	validation_0-rmse:0.46892


,model_name,fold_1,fold_2,fold_3,mean_rmsle,rmsle_std
0,xgboost_baseline,0.482901,0.491692,0.455477,0.47669,0.018889


In [ ]:
xgb_boost_eng_features_importance_df = pd.DataFrame({
    'features': xgb_eng_features,
    'importance_val1': features_import_xgb_eng[0],
    'importance_val2': features_import_xgb_eng[1],
    'importance_val3': features_import_xgb_eng[2]
})
xgb_boost_eng_features_importance_df['mean_importance_xgb'] = (xgb_boost_eng_features_importance_df['importance_val1'] + xgb_boost_eng_features_importance_df['importance_val2'] + xgb_boost_eng_features_importance_df['importance_val3'])/3
xgb_boost_eng_features_importance_df = xgb_boost_eng_features_importance_df.sort_values('mean_importance_xgb', ascending=False)
xgb_boost_eng_features_importance_df['xgb_imp_ranks'] = xgb_boost_eng_features_importance_df['mean_importance_xgb'].rank(ascending=False)

In [ ]:
quality_journal = pd.concat([catboost_quality, lightgbm_quality, xgboost_quality,
                             catboost_eng_quality, lightgbm_eng_quality, xgboost_eng_quality])
quality_journal

NameError: name 'pd' is not defined

In [ ]:
quality_journal.to_csv('../data/journals/models_quality.csv')

In [ ]:
mean_ranks_by_boosts = cat_boost_eng_features_importance_df[['features', 'cat_imp_ranks']].merge(
    right=xgb_boost_eng_features_importance_df[['features', 'xgb_imp_ranks']], on='features').merge(
    right=lgb_boost_eng_features_importance_df[['features', 'lgb_imp_ranks']], on='features'
)
mean_ranks_by_boosts['mean_all_ranks'] = (mean_ranks_by_boosts['cat_imp_ranks'] + mean_ranks_by_boosts['xgb_imp_ranks'] + mean_ranks_by_boosts['lgb_imp_ranks']) /3
mean_ranks_by_boosts = mean_ranks_by_boosts.sort_values('mean_all_ranks')
mean_ranks_by_boosts

,features,cat_imp_ranks,xgb_imp_ranks,lgb_imp_ranks,mean_all_ranks
0,full_sq,1.0,6.0,2.0,3.000000
4,sub_area,5.0,16.0,1.0,7.333333
21,non_living_sq,22.0,15.0,6.0,14.333333
14,life_sq,15.0,42.0,5.0,20.666667
26,cafe_count_5000,27.0,12.0,65.0,34.666667
...,...,...,...,...,...
393,incineration_raion,395.0,365.5,359.0,373.166667
388,baths_share,395.0,365.5,359.0,373.166667
394,retail_trade_turnover_growth,395.0,365.5,359.0,373.166667
401,provision_retail_space_modern_sqm,395.0,365.5,359.0,373.166667


In [ ]:
import json
top300_features = mean_ranks_by_boosts['features'].head(300).to_list()
with open("../data/features/features.json", "w", encoding="utf-8") as f:
    json.dump(top300_features, f, ensure_ascii=False, indent=4)

In [ ]:
oof_pred_catboost = oof_pred_catboost.reset_index(drop=True)
oof_pred_xgboost = oof_pred_xgboost.reset_index(drop=True)
oof_pred_lightgbm = oof_pred_lightgbm.reset_index(drop=True)
pred_journal = (
    oof_pred_catboost
    .merge(
        oof_pred_xgboost[["id", "fold", "val_block", "xgboost_baseline_pred", "xgboost_baseline_error"]],
        on=["id", "fold", "val_block"],
        how="left"
    )
    .merge(
        oof_pred_lightgbm[["id", "fold", "val_block", "lightgbm_baseline_pred", "lightgbm_baseline_error"]],
        on=["id", "fold", "val_block"],
        how="left"
    )
)

In [ ]:
pred_journal.to_csv('../data/journals/models_preds.csv')

In [ ]:
error_cols = ['xgboost_baseline_error', 'catboost_baseline_error', 'lightgbm_baseline_error']
error_corr_by_fold = {}

for fold in sorted(pred_journal["fold"].unique()):
    fold_df = pred_journal[pred_journal["fold"] == fold]

    error_corr_by_fold[fold] = fold_df[error_cols].corr(method="pearson")